In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import json
import time
import re
from typing import Dict, List, Optional

In [3]:
class XueQiuFinanceScraper:
    """雪球财务数据爬虫类"""
    
    def __init__(self):
        self.base_url = "https://xueqiu.com"
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
            'Accept-Language': 'zh-CN,zh;q=0.9,en;q=0.8',
            'Accept-Encoding': 'gzip, deflate, br',
            'Connection': 'keep-alive',
            'Upgrade-Insecure-Requests': '1',
        }
        self.session = requests.Session()
    
    def _get_cookies(self) -> None:
        """获取雪球网站的cookies"""
        try:
            response = self.session.get(self.base_url, headers=self.headers, timeout=10)
            response.raise_for_status()
        except Exception as e:
            print(f"获取cookies失败: {e}")
    
    def get_stock_finance_data(self, stock_code: str) -> Dict:
        """
        获取股票的财务数据
        
        Args:
            stock_code: 股票代码，例如 "SH600489"
            
        Returns:
            包含财务数据的字典
        """
        try:
            # 先获取cookies
            self._get_cookies()
            
            # 构建股票详情页URL
            url = f"{self.base_url}/S/{stock_code}"
            
            # 获取股票详情页
            response = self.session.get(url, headers=self.headers, timeout=15)
            response.raise_for_status()
            
            # 解析HTML
            soup = BeautifulSoup(response.text, 'html.parser')
            
            # 提取财务数据
            finance_data = self._parse_finance_data(soup, stock_code)
            
            return finance_data
            
        except Exception as e:
            print(f"获取财务数据失败: {e}")
            return {}
    
    def _parse_finance_data(self, soup: BeautifulSoup, stock_code: str) -> Dict:
        """解析财务数据"""
        finance_data = {
            'stock_code': stock_code,
            'basic_info': {},
            'financial_indicators': {},
            'market_data': {},
            'company_info': {}
        }
        
        try:
            # 1. 提取基本信息
            finance_data['basic_info'] = self._extract_basic_info(soup)
            
            # 2. 提取财务指标
            finance_data['financial_indicators'] = self._extract_financial_indicators(soup)
            
            # 3. 提取市场数据
            finance_data['market_data'] = self._extract_market_data(soup)
            
            # 4. 提取公司信息
            finance_data['company_info'] = self._extract_company_info(soup)
            
            # 5. 尝试通过API获取更详细的财务数据
            detailed_data = self._get_detailed_finance_api(stock_code)
            if detailed_data:
                finance_data.update(detailed_data)
                
        except Exception as e:
            print(f"解析财务数据失败: {e}")
        
        return finance_data
    
    def _extract_basic_info(self, soup: BeautifulSoup) -> Dict:
        """提取基本信息"""
        basic_info = {}
        
        try:
            # 股票名称
            name_tag = soup.find('h1', class_='stock-name')
            if name_tag:
                basic_info['stock_name'] = name_tag.get_text().strip()
            
            # 当前价格
            price_tag = soup.find('div', class_='stock-price')
            if price_tag:
                price_text = price_tag.get_text().strip()
                match = re.search(r'[\d.]+', price_text)
                if match:
                    basic_info['current_price'] = float(match.group())
            
            # 涨跌幅
            change_tag = soup.find('div', class_='stock-change')
            if change_tag:
                change_text = change_tag.get_text().strip()
                basic_info['change_info'] = change_text
                
        except Exception as e:
            print(f"提取基本信息失败: {e}")
        
        return basic_info
    
    def _extract_financial_indicators(self, soup: BeautifulSoup) -> Dict:
        """提取财务指标"""
        indicators = {}
        
        try:
            # 查找包含财务指标的元素
            indicator_tags = soup.find_all('div', class_=re.compile('indicator|finance'))
            
            for tag in indicator_tags:
                text = tag.get_text()
                # EPS (每股收益)
                if '每股收益' in text or 'EPS' in text:
                    match = re.search(r'[\d.]+', text)
                    if match:
                        indicators['eps'] = float(match.group())
                
                # 每股净资产
                if '每股净资产' in text or 'BVPS' in text:
                    match = re.search(r'[\d.]+', text)
                    if match:
                        indicators['bvps'] = float(match.group())
                
                # 股息
                if '股息' in text:
                    match = re.search(r'[\d.]+', text)
                    if match:
                        indicators['dividend'] = float(match.group())
                
                # 股息率
                if '股息率' in text:
                    match = re.search(r'[\d.]+', text)
                    if match:
                        indicators['dividend_yield'] = float(match.group())
                
                # 市盈率
                if '市盈率' in text or 'PE' in text:
                    match = re.search(r'[\d.]+', text)
                    if match:
                        indicators['pe_ratio'] = float(match.group())
                
                # 市净率
                if '市净率' in text or 'PB' in text:
                    match = re.search(r'[\d.]+', text)
                    if match:
                        indicators['pb_ratio'] = float(match.group())
                        
        except Exception as e:
            print(f"提取财务指标失败: {e}")
        
        return indicators
    
    def _extract_market_data(self, soup: BeautifulSoup) -> Dict:
        """提取市场数据"""
        market_data = {}
        
        try:
            # 总股本
            if '总股本' in soup.get_text():
                match = re.search(r'总股本[\s:：]*(\d+\.?\d*)[亿股]*', soup.get_text())
                if match:
                    market_data['total_shares'] = float(match.group(1))
            
            # 流通股
            if '流通股' in soup.get_text():
                match = re.search(r'流通股[\s:：]*(\d+\.?\d*)[亿股]*', soup.get_text())
                if match:
                    market_data['float_shares'] = float(match.group(1))
            
            # 总市值
            if '总市值' in soup.get_text():
                match = re.search(r'总市值[\s:：]*(\d+\.?\d*)[亿元]*', soup.get_text())
                if match:
                    market_data['market_cap'] = float(match.group(1))
                    
        except Exception as e:
            print(f"提取市场数据失败: {e}")
        
        return market_data
    
    def _extract_company_info(self, soup: BeautifulSoup) -> Dict:
        """提取公司信息"""
        company_info = {}
        
        try:
            # 公司网站
            website_tag = soup.find('a', href=re.compile('http'))
            if website_tag:
                company_info['website'] = website_tag['href']
            
            # 公司地址和电话
            text = soup.get_text()
            # 地址
            address_match = re.search(r'[地址|Address][\s:：]*(.+?)(?=\d{11}|电话|Phone)', text)
            if address_match:
                company_info['address'] = address_match.group(1).strip()
            
            # 电话
            phone_match = re.search(r'(?:电话|Phone|Tel)[\s:：]*(\d{3,4}-\d{7,8})', text)
            if phone_match:
                company_info['phone'] = phone_match.group(1)
                
        except Exception as e:
            print(f"提取公司信息失败: {e}")
        
        return company_info
    
    def _get_detailed_finance_api(self, stock_code: str) -> Optional[Dict]:
        """通过雪球API获取详细财务数据"""
        try:
            # 雪球财务数据API
            symbol = stock_code.replace('SH', '').replace('SZ', '')
            exchange = 'SH' if stock_code.startswith('SH') else 'SZ'
            
            api_url = f"https://stock.xueqiu.com/v5/stock/finance/cn/balance.json"
            params = {
                'symbol': f'{exchange}{symbol}',
                'type': 'all',
                'is_detail': 'true',
                'count': '10'
            }
            
            response = self.session.get(api_url, headers=self.headers, params=params, timeout=10)
            
            if response.status_code == 200:
                data = response.json()
                if data.get('error_code') == 0:
                    return {
                        'detailed_finance': data.get('data', {})
                    }
                    
        except Exception as e:
            print(f"获取API财务数据失败: {e}")
        
        return None
    
    def get_all_financial_statements(self, stock_code: str) -> Dict:
        """
        获取完整的财务报表（资产负债表、利润表、现金流量表）
        
        Args:
            stock_code: 股票代码
            
        Returns:
            包含三大财务报表的字典
        """
        statements = {
            'balance_sheet': [],      # 资产负债表
            'income_statement': [],   # 利润表
            'cash_flow_statement': [] # 现金流量表
        }
        
        try:
            symbol = stock_code.replace('SH', '').replace('SZ', '')
            exchange = 'SH' if stock_code.startswith('SH') else 'SZ'
            
            # 1. 资产负债表
            balance_url = f"https://stock.xueqiu.com/v5/stock/finance/cn/balance.json"
            balance_params = {
                'symbol': f'{exchange}{symbol}',
                'type': 'all',
                'is_detail': 'true',
                'count': '10'
            }
            balance_resp = self.session.get(balance_url, headers=self.headers, params=balance_params)
            if balance_resp.status_code == 200:
                balance_data = balance_resp.json()
                if balance_data.get('data'):
                    statements['balance_sheet'] = balance_data['data'].get('list', [])
            
            time.sleep(0.5)  # 避免请求过快
            
            # 2. 利润表
            income_url = f"https://stock.xueqiu.com/v5/stock/finance/cn/income.json"
            income_params = {
                'symbol': f'{exchange}{symbol}',
                'type': 'all',
                'is_detail': 'true',
                'count': '10'
            }
            income_resp = self.session.get(income_url, headers=self.headers, params=income_params)
            if income_resp.status_code == 200:
                income_data = income_resp.json()
                if income_data.get('data'):
                    statements['income_statement'] = income_data['data'].get('list', [])
            
            time.sleep(0.5)
            
            # 3. 现金流量表
            cashflow_url = f"https://stock.xueqiu.com/v5/stock/finance/cn/cash_flow.json"
            cashflow_params = {
                'symbol': f'{exchange}{symbol}',
                'type': 'all',
                'is_detail': 'true',
                'count': '10'
            }
            cashflow_resp = self.session.get(cashflow_url, headers=self.headers, params=cashflow_params)
            if cashflow_resp.status_code == 200:
                cashflow_data = cashflow_resp.json()
                if cashflow_data.get('data'):
                    statements['cash_flow_statement'] = cashflow_data['data'].get('list', [])
                    
        except Exception as e:
            print(f"获取财务报表失败: {e}")
        
        return statements


def get_xueqiu_finance_data(stock_code: str = "SH600489") -> Dict:
    """
    获取雪球网站股票的全部财务数据
    
    Args:
        stock_code: 股票代码，默认为中金黄金(SH600489)
        
    Returns:
        包含所有财务数据的字典
    """
    scraper = XueQiuFinanceScraper()
    
    print(f"正在获取 {stock_code} 的财务数据...")
    
    # 获取基本信息和财务指标
    basic_data = scraper.get_stock_finance_data(stock_code)
    
    # 获取详细财务报表
    # statements = scraper.get_all_financial_statements(stock_code)
    
    # 合并数据
    # result = {**basic_data, **statements}
    result = {**basic_data}
    
    print(f"财务数据获取完成！")
    
    return result


def save_finance_data_to_excel(data: Dict, filename: str = "finance_data.xlsx"):
    """
    将财务数据保存到Excel文件
    
    Args:
        data: 财务数据字典
        filename: Excel文件名
    """
    try:
        with pd.ExcelWriter(filename, engine='openpyxl') as writer:
            # 基本信息
            if data.get('basic_info'):
                basic_df = pd.DataFrame([data['basic_info']])
                basic_df.to_excel(writer, sheet_name='基本信息', index=False)
            
            # 财务指标
            if data.get('financial_indicators'):
                indicators_df = pd.DataFrame([data['financial_indicators']])
                indicators_df.to_excel(writer, sheet_name='财务指标', index=False)
            
            # 市场数据
            if data.get('market_data'):
                market_df = pd.DataFrame([data['market_data']])
                market_df.to_excel(writer, sheet_name='市场数据', index=False)
            
            # 公司信息
            if data.get('company_info'):
                company_df = pd.DataFrame([data['company_info']])
                company_df.to_excel(writer, sheet_name='公司信息', index=False)
            
            # 资产负债表
            if data.get('balance_sheet'):
                balance_df = pd.DataFrame(data['balance_sheet'])
                balance_df.to_excel(writer, sheet_name='资产负债表', index=False)
            
            # 利润表
            if data.get('income_statement'):
                income_df = pd.DataFrame(data['income_statement'])
                income_df.to_excel(writer, sheet_name='利润表', index=False)
            
            # 现金流量表
            if data.get('cash_flow_statement'):
                cashflow_df = pd.DataFrame(data['cash_flow_statement'])
                cashflow_df.to_excel(writer, sheet_name='现金流量表', index=False)
        
        print(f"财务数据已保存到 {filename}")
        
    except Exception as e:
        print(f"保存Excel失败: {e}")

In [5]:
get_xueqiu_finance_data("SZ000001")

正在获取 SZ000001 的财务数据...
财务数据获取完成！


{'stock_code': 'SZ000001',
 'basic_info': {},
 'financial_indicators': {},
 'market_data': {},
 'company_info': {}}